In [1]:
import jax
import jax.numpy as jnp

In [2]:
def assert_sym(M):
    assert jnp.all(M == M.T) or jnp.all(jnp.isclose(M, M.T, atol=1e-6)), "Not symmmetric!"

def symmetrize(M):
    return 0.5 * (M+M.T)

def assert_pd(M):
    M = symmetrize(M)
    assert jnp.all(jnp.linalg.eigvalsh(M) > 0), "Not PD!"
    
def assert_spd(M):
    assert_sym(M)
    assert_pd(M)

In [29]:
_key = lambda x: jax.random.PRNGKey(x)
def random_spd(key, d, /):
    Eps = jax.random.normal(key, (d,d))
    return Eps @ Eps.T# + jnp.eye(d)

D = 100

m = jax.random.normal(_key(112), (1,D))
C = random_spd(_key(911), D)  
A = random_spd(_key(114), D)  
# B = random_spd(_key(1813), D) 
B = jnp.eye(D) * .5

assert_spd(A)
assert_spd(B)
assert_spd(C)

C_sqrt = jnp.linalg.cholesky(C, upper=True)
B_sqrt = jnp.linalg.cholesky(B, upper=True)

sampling_dist = jax.random.rademacher
# sampling_dist = jax.random.normal

def sample_x(num_samples, /, key):
    N = sampling_dist(key=key, shape=(num_samples, D))
    return N@C_sqrt + m


def sample_y_given_x(x0, /, key):
    num_samples = x0.shape[0]
    N = sampling_dist(key=key, shape=(num_samples, D))
    return N@B_sqrt + x0@A


meow = C@A.T
meow2 = (A @ meow + B)
def K_apply_dense(v):
    return meow @ jax.scipy.linalg.solve(meow2.T, v.T)


y = jnp.zeros(1)                                                # TODO What is y? Synthetic data?
def sample_x_given_y(num_samples, /, key):
    key_x, key_y = jax.random.split(key)
    x0 = sample_x(num_samples, key=key_x)
    y0 = sample_y_given_x(x0, key=key_y)
    w  = K_apply_dense(y0 - y)
    return x0 - w


In [30]:
from matfree.lstsq import lsmr

def meow_apply(v):
    return meow2 @ v

solve = lsmr(atol=1e-3,btol=1e-3,ctol=1e-4)
def K_apply_mf(Vs):
    def body(v):
        x, info = solve(meow_apply, v)
        return meow @ x
    return jax.vmap(body)(Vs)
    # return jax.lax.map(body, Vs)

def sample_x_given_y_mf(num_samples, /, key):
    key_x, key_y = jax.random.split(key)
    x0 = sample_x(num_samples, key=key_x)
    y0 = sample_y_given_x(x0, key=key_y)
    w = K_apply_mf(y0 - y)
    return x0 - w

In [31]:
a_lot_of_samples = sample_x_given_y_mf(1_000, _key(117))

In [32]:
# analytical_mean = m - K_apply_dense((m@A - y).squeeze())
analytical_mean = m - K_apply_mf((m@A - y))
print("======MEAN======")
print(a_lot_of_samples.mean(0))
print(analytical_mean.squeeze())

print("======VAR=======")
# analytical_covariance = C - K_apply_dense(meow.squeeze())
analytical_covariance = C - K_apply_mf(meow)
print(a_lot_of_samples.var(0))
print(jnp.diag(analytical_covariance))

======MEAN======
[ 0.16528827 -1.3042562   1.1946949  -1.2341894   0.8127644  -0.914277
 -0.13905655  0.95365024  1.2383221  -1.2252067  -0.07074793 -1.498625
  0.8319095   2.67297    -0.45212072 -0.4668737   0.34253353 -1.3562733
 -1.6635165  -1.6302212   2.57761     0.2200013   0.8750166   0.5558928
 -0.53829646  0.16399404 -0.6131358   0.97484946  0.2032652  -0.9416656
 -0.7006276  -0.01792208 -1.6108111  -0.8213546  -0.17959674 -0.7630917
  0.89331424  1.2936957  -1.3022856  -1.6876373  -0.03197263 -0.59538674
  2.063702   -0.7691982  -0.05500199  1.5736626  -0.18242967 -0.19512782
 -1.4970931   0.28383827 -0.97530395 -0.91544116  0.22616532  1.1180086
 -0.70184594 -0.33472523  1.6237484   1.2256325  -0.32002985 -0.72471887
  0.92419934 -1.4770712  -1.3176097  -0.05188543 -0.4162493   0.37428626
  0.88068134  0.22112298  0.10370276 -1.4384583  -0.51940495 -0.93006456
  0.36350238  1.4337724   0.53191626 -0.18509065 -2.1242878  -2.162324
  1.3145545   1.4557174   0.5376474  -0.65739

In [33]:
t1 = (a_lot_of_samples - a_lot_of_samples.mean(0, keepdims=True))
sample_cov = t1.T@t1 / (a_lot_of_samples.shape[0] - 1)

jnp.linalg.trace(sample_cov), jnp.linalg.trace(analytical_covariance)

(Array(1646.8164, dtype=float32), Array(1887.5214, dtype=float32))

In [34]:
S = random_spd(_key(72), D)
Si = jnp.linalg.inv(S)

jnp.linalg.trace(Si@analytical_covariance)

Array(3820.4028, dtype=float32)

In [35]:
from jax.flatten_util import ravel_pytree
from matfree import stochtrace

problem = stochtrace.integrand_trace()

def sampler_lla(*args_like, num):
    # TODO pass a bunch of stuff that makes sense in the LLA / GGN context
    x_flat, unflatten = ravel_pytree(*args_like)
    def sampler(key):
        return sample_x_given_y_mf(num, key)
    return sampler


sampler = sampler_lla(m, num=10_000)
estimate = stochtrace.estimator(problem, sampler)
estimate = jax.jit(estimate, static_argnums=[0])
estimate(lambda v: Si@v, key=_key(1235))

Array(3581.3647, dtype=float32)

In [36]:
sampler = stochtrace.sampler_normal(jnp.ones(D), num=10_000)
estimate = stochtrace.estimator(problem, sampler)
def Si_apply(v):
    x,info = jax.scipy.sparse.linalg.cg(S, v)
    return x
estimate(lambda v: analytical_covariance@Si_apply(v), key=_key(123))
# estimate(lambda v: Si@v, key=_key(123))

Array(3854.2356, dtype=float32)